# 繼續預訓練 BERT：用遮蔽語言模型（MLM）適應影評領域
---

## 這份程式要做什麼

讓 BERT 在**影評資料**上繼續玩「填空遊戲」，使它更熟悉影評的用語。

```
bert-base-uncased（讀過維基百科與書籍，懂一般英文）
        │
        ▼  在 Rotten Tomatoes 影評上繼續做「遮蔽詞預測」訓練
        │
MLM_Finetune_model（更熟悉影評用語的 BERT）
```

最後用同一句 `"What a horrible [MASK]."` 對照訓練前後的填空結果。

## 什麼是 MLM（Masked Language Modeling，遮蔽語言模型）

把句子裡**隨機一部分詞蓋住**，讓模型猜被蓋住的詞是什麼：

```
原句：  What a horrible movie .
遮蔽：  What a horrible [MASK] .
目標：  模型要預測 [MASK] 是 "movie"
```

這正是 BERT 當初**預訓練時用的任務**之一，所以我們現在是**用同樣的任務，在新領域的文字上繼續訓練**。

## 為什麼要這樣做：領域適應（domain adaptation）

| 問題 | 說明 |
|---|---|
| BERT 沒看過多少影評用語 | 影評有自己的口吻與詞彙（"a stunning debut"、"a snooze fest"） |
| 解法 | 用影評文字繼續做 MLM，**不需要任何標籤** |
| 好處 | 之後再拿這個模型去做影評分類，通常比直接用原版 BERT 更好 |

## 和前面學過的方法比較

| 方法 | 需要標籤嗎 | 模型輸出 | 目標 |
|---|---|---|---|
| 一般微調（Trainer 分類） | 要 | 類別 | 判斷影評正負面 |
| SetFit | 要（少量） | 類別 | 少樣本分類 |
| TSDAE | 不要 | **句向量** | 弄壞句子 → 還原，學出好的句向量 |
| **MLM（這份）** | **不要** | **每個詞的預測** | 猜被遮住的詞，讓模型更熟悉領域語言 |

TSDAE 和 MLM 都是「弄壞再還原」，差別在於：TSDAE 的目標是**整句的向量**，MLM 的目標是預測**被遮住的單一個詞**。

## 流程總覽

```
載入資料 → 載入模型與分詞器 → 分詞 → 建立遮蔽器(collator) → 設定參數 → 建立 Trainer
     → 訓練前填空測試 → 訓練並儲存 → 訓練後填空測試
```

## 步驟 0：載入套件

| 匯入 | 用途 |
|---|---|
| `load_dataset` | 載入 Hugging Face 資料集 |
| `AutoTokenizer` | 載入配對的分詞器 |
| `AutoModelForMaskedLM` | 載入「BERT + 遮蔽詞預測頭」的模型 |
| `TrainingArguments`、`Trainer` | 設定訓練參數、執行訓練 |
| `DataCollatorForLanguageModeling` | **負責隨機遮蔽詞**，並組成 batch |
| `pipeline` | 一行程式就能使用模型（這裡用來做填空） |

名稱裡的 `ForMaskedLM` 表示模型最上面接的是「遮蔽詞預測頭」，輸出的是**詞彙表上每個詞的分數**，不是上一份的 2 類分類。

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMaskedLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling
from transformers import pipeline

## 步驟 1：載入資料

```python
train_data, test_data = tomatoes["train"], tomatoes["test"]
```

Rotten Tomatoes 的每筆資料有 `text`（一句影評）和 `label`（正負面）。這裡取 train 和 test 兩份。

**MLM 用不到 label**，只用 `text`：因為 MLM 的「答案」就是句子裡被遮住的詞本身。

In [ ]:
tomatoes = load_dataset("cornell-movie-review-data/rotten_tomatoes")
train_data, test_data = tomatoes["train"], tomatoes["test"]

## 步驟 2：載入模型與分詞器

| 部分 | 說明 |
|---|---|
| `bert-base-uncased` | **不分大小寫**的 BERT（上一份影評分類用的是 `cased`） |
| `AutoModelForMaskedLM` | 載入時**不需要 `num_labels`**：預訓練 BERT 本來就帶著遮蔽詞預測頭，不是新加的 |
| `AutoTokenizer` | 和模型配對的分詞器，所以都用同一個 `model_id` |

之前微調分類時，分類頭是新加的、隨機初始化的；這裡的預測頭是**原本就訓練好的**，我們只是在影評文字上繼續訓練它。

In [ ]:
model_id = "bert-base-uncased"
model = AutoModelForMaskedLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

## 步驟 3：分詞並移除 label

| 部分 | 說明 |
|---|---|
| `lambda x: tokenizer(x["text"], truncation=True)` | 丟一批影評進去，吐出分詞後的結果；`truncation=True` 超過最大長度就截斷 |
| `batched=True` | 一次處理一批，比逐筆快 |
| `.remove_columns(["label"])` | MLM 不需要標籤，移除掉 |

### 這個 lambda 怎麼讀

```python
lambda x: tokenizer(x["text"], truncation=True)
```

等價於：

```python
def f(x):
    return tokenizer(x["text"], truncation=True)
```

`batched=True` 時，`x` 是一批資料，`x["text"]` 是句子清單。

### 沒有補齊（padding）

句子長短不一，但分詞這一步**不補齊**，補齊交給下一步的 collator 在組 batch 時處理（只補到該 batch 最長的那句，比較省）。

`text` 欄位沒有被移除，但 `Trainer` 預設會自動丟掉模型用不到的欄位（這是我依 Trainer 的預設行為推斷的，沒有針對你的版本確認）。

In [ ]:
tokenized_train_data = train_data.map(
    lambda x: tokenizer(x["text"], truncation=True),
    batched=True
).remove_columns(["label"])

tokenized_test_data = test_data.map(
    lambda x: tokenizer(x["text"], truncation=True),
    batched=True
).remove_columns(["label"])

## 步驟 4：建立資料整理器 —— 遮蔽詞就在這裡發生

這是整份程式**最關鍵**的地方：**遮蔽不是事先做好的，而是在每個 batch 組成時才隨機做。**

| 參數 | 說明 |
|---|---|
| `mlm=True` | 使用遮蔽語言模型（`False` 則是 GPT 那種預測下一個詞） |
| `mlm_probability=0.15` | 每個 token 有 15% 的機率被選中來預測 |

### 被選中的 token 怎麼處理

依 BERT 論文的做法，我記得是：

| 比例 | 處理 |
|---|---|
| 80% | 換成 `[MASK]` |
| 10% | 換成隨機的另一個詞 |
| 10% | 保持原樣 |

這樣做是為了讓模型不要只學會「看到 `[MASK]` 才預測」。這個比例我沒有針對你的版本確認。

### 它還產生 `labels`

被選中的位置放**原本的詞**（答案），其他位置放 `-100`，loss 計算時會被忽略。所以模型只對被遮住的位置算損失。

### 動態遮蔽

因為每個 batch 都重新隨機遮蔽，**同一句影評在不同 epoch 會被遮住不同的位置**，等於讓資料更多樣。collator 同時負責動態補齊。

```
原句   what a horrible movie .
第1輪  what a [MASK] movie .
第2輪  what a horrible [MASK] .
第3輪  [MASK] a horrible movie .
```

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

## 步驟 5：訓練參數

| 參數 | 值 | 說明 |
|---|---|---|
| `output_dir` | `"MLM_Finetune_model"` | 輸出資料夾 |
| `num_train_epochs` | 10 | 整份資料看 10 遍 |
| `learning_rate` | `2e-5` | 小學習率，繼續預訓練只需小幅調整 |
| `per_device_train_batch_size` | 16 | 每步 16 句 |
| `per_device_eval_batch_size` | 16 | 評估時每次 16 句 |
| `weight_decay` | 0.01 | 輕微的正則化 |
| `save_strategy` | `"epoch"` | 每個 epoch 存一次 checkpoint |
| `report_to` | `"none"` | 不傳記錄到外部工具 |

步數速算：8530 筆 ÷ 16 ≈ 534 步 / epoch，10 個 epoch ≈ 5340 步。

In [ ]:
args = TrainingArguments(
    output_dir="MLM_Finetune_model",
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none",
)

## 步驟 6：建立 Trainer

`Trainer` 把所有零件組裝在一起：

| 參數 | 說明 |
|---|---|
| `model` | 要訓練的 BERT（含遮蔽詞預測頭） |
| `args` | 訓練參數 |
| `train_dataset` | 分詞後的訓練資料 |
| `eval_dataset` | 分詞後的測試資料 |
| `data_collator` | 負責遮蔽與補齊 |

沒有 `compute_metrics`：MLM 通常看 loss（或由 loss 算出的困惑度）就好，不像分類要算 F1。

> **建立 Trainer 不會自動開始訓練**，要呼叫 `trainer.train()`。另外，這裡沒有設定 `eval_strategy`，預設訓練中不評估，所以 `eval_dataset` 目前其實沒有被用到。

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train_data,
    eval_dataset=tokenized_test_data,
    data_collator=data_collator,
)

## 步驟 7：定義填空測試函式 `prediction`

這個函式會被呼叫兩次（訓練前、訓練後），所以保留成函式。

```python
mask_filler = pipeline("fill-mask", model=model)
preds = mask_filler("What a horrible [MASK].")
```

| 部分 | 說明 |
|---|---|
| `pipeline("fill-mask", ...)` | 做填空任務的現成工具 |
| `model=model` | 可以是**模型名稱字串或資料夾路徑**，pipeline 會自己載入 |
| `[MASK]` | `bert-base-uncased` 使用的遮蔽符號，不同模型的符號不一樣 |
| 回傳 `preds` | 回傳前 5 名，每個是 dict，含 `sequence`（填完的句子）、`score`（機率）等 |
| `{pred["score"] : .4f}` | 取 4 位小數；冒號後的空格會讓正數前面多一個空格 |

### 參數名稱 `model` 與外面的變數 `model` 同名

函式的參數 `model` 是**區域變數**，它會「遮住」外面那個 BERT 模型 `model`。在函式裡看到的 `model`，永遠是呼叫時傳進來的那個東西。不衝突，但容易混淆。

### 傳字串 vs 傳模型物件

| 傳入 | pipeline 做什麼 |
|---|---|
| `"bert-base-uncased"`（字串） | 重新載入一份**原版模型** |
| `"MLM_Finetune_model"`（資料夾路徑） | 載入**存好的訓練後模型** |
| `model`（記憶體中的物件） | 直接使用，**訓練後就是訓練過的版本**（訓練是原地修改） |

> **Python 版本提醒**：f-string 裡又用了同一種雙引號（`f"{pred["sequence"]}"`），這個寫法需要 **Python 3.12 以上**。舊版本會 `SyntaxError`，改成外層雙引號、內層單引號（`pred['sequence']`）即可。

In [ ]:
def prediction(model):
    mask_filler = pipeline(
        "fill-mask",
        model=model,
    )
    preds = mask_filler("What a horrible [MASK].")

    for pred in preds:
        print(f"{pred["sequence"]}, Score : {pred["score"] : .4f}")

## 步驟 8：訓練前的填空測試（基準線）

傳入字串 `model_id`，pipeline 會載入**原版**的 `bert-base-uncased`。

這是對照用的基準：看原版 BERT 對 `"What a horrible [MASK]."` 會填什麼。預測偏向一般英文用語是正常的。

In [ ]:
print("Before Training : \n")
prediction(model = model_id)

## 步驟 9：訓練並儲存（真正的訓練在這裡）

| 步驟 | 為什麼 |
|---|---|
| `tokenizer.save_pretrained(args.output_dir)` | 先把分詞器存進輸出資料夾。之後用 `pipeline` 從資料夾載入模型時，同一個資料夾要找得到分詞器 |
| `trainer.train()` | 開始訓練，**原地更新 `model` 的權重** |
| `model.save_pretrained(args.output_dir)` | 把訓練好的權重存進資料夾**根目錄**，讓它可以被直接載入 |

### `trainer.train()` 內部每個 batch 做的事

```
1. 取 16 句影評，collator 隨機遮蔽約 15% 的 token
2. BERT 預測被遮住位置的詞
3. 和原本的詞計算 loss（交叉熵）
4. 反向傳播，優化器更新權重
```

### 要跑多久

10 個 epoch，約 5340 步，需要一些時間，建議有 GPU。

### 關於資料夾

`output_dir` 同時放了每個 epoch 的 checkpoint 和最後存的最終模型。這樣能用，只是較雜；而且 10 份 checkpoint 很佔空間（每份除了模型權重還有優化器狀態，我粗估單份可能超過 1 GB，沒有實測）。記得把 `MLM_Finetune_model/` 加進 `.gitignore`。

In [ ]:
tokenizer.save_pretrained(args.output_dir)
trainer.train()
model.save_pretrained(args.output_dir)

## 步驟 10：訓練後的填空測試

傳入剛存好的資料夾 `"MLM_Finetune_model"`，pipeline 會載入**訓練後**的模型。

和上面訓練前的結果對照，看影評領域的訓練有沒有改變預測。

In [ ]:
print("\n After Training : \n")
prediction(model = "MLM_Finetune_model")

## 怎麼解讀結果

| 對照 | 要看什麼 |
|---|---|
| 訓練前（原版 BERT） | 預測的詞偏向一般英文，可能和影評無關 |
| 訓練後 | 預期更常出現影評領域的詞，例如 `movie`、`film` 這類（我沒有實測，以你跑出來的為準） |

注意：只用**一個句子**看效果非常有限，建議多換幾個句子，例如 `"A beautifully shot and [MASK] film."`。

---

# 重點整理

## 一、整份程式一句話

> 載入 `bert-base-uncased`，在 Rotten Tomatoes 影評上以「隨機遮住 15% 的詞、讓模型猜回來」的任務繼續訓練 10 個 epoch，並用填空測試對照訓練前後。

## 二、必須搞懂的 8 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | MLM | 遮住部分詞，讓模型猜；BERT 預訓練的核心任務 |
| 2 | 領域適應 | 用目標領域的文字繼續預訓練，讓模型更熟悉該領域 |
| 3 | 非監督 | 只需要文字，不需要標籤（所以 `remove_columns(["label"])`） |
| 4 | `[MASK]` | BERT 的遮蔽符號 |
| 5 | `mlm_probability=0.15` | 每個 token 有 15% 機率被選中來預測 |
| 6 | 動態遮蔽 | 遮蔽在每個 batch 才隨機做，不同 epoch 遮住的位置不同 |
| 7 | `labels = -100` | collator 產生的標籤中，沒被遮住的位置用 -100，loss 會忽略 |
| 8 | `pipeline("fill-mask")` | 現成的填空工具，`model` 可以是名稱或資料夾路徑 |

## 三、程式流程速記

```
1. load_dataset            → 取 train / test
2. from_pretrained         → bert-base-uncased + 遮蔽詞預測頭 + 分詞器
3. map(tokenizer)          → 分詞（截斷不補齊），移除 label
4. DataCollatorForLM       → 設定「遮蔽 15%」，同時負責動態補齊
5. TrainingArguments       → 10 epochs、學習率 2e-5、batch 16
6. Trainer                 → 組裝零件
7. prediction（訓練前）    → 原版 BERT 的填空結果
8. train + save            → 先存 tokenizer → 訓練 → 存模型
9. prediction（訓練後）    → 訓練後模型的填空結果
```

## 四、最容易搞混的地方

1. **為什麼不用 `num_labels`？**
   預訓練 BERT 本來就帶遮蔽詞預測頭，不是新加的。

2. **為什麼要移除 label？**
   MLM 的「答案」就是句子本身被遮住的詞，不需要影評的正負面標籤。

3. **遮蔽是在哪裡做的？**
   不是在分詞時，而是在 `DataCollatorForLanguageModeling` 組 batch 時動態做。

4. **為什麼 `prediction` 傳字串就能得到原版 BERT？**
   傳字串時，pipeline 會自己重新載入；傳記憶體中的 `model`，訓練後就是訓練過的版本。

5. **函式參數 `model` 和外面的 `model` 是同一個嗎？**
   不是。函式裡的 `model` 是區域變數，只代表呼叫時傳進來的東西。

6. **為什麼要先存 tokenizer？**
   之後 `pipeline` 從 `"MLM_Finetune_model"` 載入模型時，同一個資料夾要有分詞器。

7. **MLM 和 TSDAE 有什麼不同？**
   都是「弄壞再還原」，但 MLM 預測單一被遮住的詞；TSDAE 靠一個句向量還原整句，目標是句向量。

8. **這個模型能直接分類嗎？**
   不能，它只會「填空」。要分類，還需要再接分類頭做微調。

## 五、觀察到的幾個地方，僅供參考

1. **`eval_dataset` 傳了，但訓練中不會評估**
   沒有設定 `eval_strategy`，預設不評估，那份測試集目前沒被用到，也沒有任何 metric。MLM 常用 `eval_loss` 或困惑度（`exp(eval_loss)`）。

2. **評估時 collator 也會隨機遮蔽**
   我推測這會讓 `eval_loss` 每次略有不同。要可重現的比較，需固定隨機種子。

3. **10 個 epoch 可能偏多**
   只有 8530 句短影評，訓練太久可能過擬合，最好觀察 `eval_loss` 是否開始上升。

4. **checkpoint 佔空間**
   每個 epoch 存一份，共 10 份，可以設 `save_total_limit` 限制保留份數。

5. **`output_dir` 與最終存檔同一個資料夾**
   checkpoint 子資料夾和最終模型放在一起，能用，只是較雜。

6. **f-string 巢狀雙引號需要 Python 3.12+**
   舊版本會 `SyntaxError`。

7. **隨機性**
   遮蔽是隨機的，沒有固定 seed 時，每次訓練結果略有不同。

8. **只用一句話測試**
   結論不夠有代表性，建議多測幾句。

## 六、可以自己做的實驗（需要時再自己加）

```python
# 1. 同時比較多個句子
sentences = [
    "What a horrible [MASK].",
    "A beautifully shot and [MASK] film.",
    "The acting was [MASK] but the plot was weak.",
]

# 2. 計算困惑度（trainer 已建立，可以直接用）
import math
metrics = trainer.evaluate()
print("perplexity:", math.exp(metrics["eval_loss"]))

# 3. 調整 mlm_probability（0.10、0.15、0.25），比較填空表現

# 4. 把 num_train_epochs 從 10 降到 3，看填空結果差多少

# 5. 最有意義的實驗：把這個「適應過影評」的模型拿去做分類，
#    跟原版 bert-base-uncased 比較 F1
from transformers import AutoModelForSequenceClassification
clf = AutoModelForSequenceClassification.from_pretrained(
    "MLM_Finetune_model", num_labels=2
)
# 接著沿用「微調 BERT 影評分類」那份程式，看 F1 有沒有比原版更好
```

## 七、各種訓練方式的定位

| 方式 | 需要標籤 | 目標 | 典型用途 |
|---|---|---|---|
| 一般微調（分類） | 要，量多 | 分類 | 資料充足 |
| SetFit | 要，量極少 | 分類 | 標註稀少 |
| TSDAE | 不要 | 句向量 | 無標註、要做語意搜尋 |
| **MLM 繼續預訓練** | **不要** | **語言理解** | **先適應領域，再接下游任務** |

典型的組合流程：**MLM 領域適應 → 再接分類頭做微調**。

## 八、一句話記住

> **MLM 繼續預訓練 = 在目標領域的文字上，讓 BERT 反覆玩「猜被遮住的詞」，不需要標籤，讓它更熟悉該領域的語言。**